In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore")

In [2]:
df_train = pd.read_csv("/kaggle/input/playground-series-s5e5/train.csv")
df_test = pd.read_csv("/kaggle/input/playground-series-s5e5/test.csv")

# Data Preprocessing

In [3]:
num_vars = df_train.drop(columns=['id','Calories']).select_dtypes(include=['int64', 'float64']).columns
cat_vars = ['Sex']

In [4]:
df_train['Sex'] = df_train['Sex'].astype('category')
df_test['Sex'] = df_test['Sex'].astype('category')

# Feature Engineering

In [5]:
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler

In [6]:
df_train.head()

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


In [7]:
def make_log_features(df):
    df_temp = df.copy()

    num_features = df_temp.drop(columns=[col for col in df_temp.columns if 'alories' in col]).select_dtypes(include=['int64', 'float64']).columns

    for feature in num_features:
        df_temp[f'log_{feature}'] = np.log(df_temp[feature]) # not robust to 0's but no 0's in data by looks of it in eda

    return df_temp

def make_interactions(df, test = False, log = False):
    df_temp = df.copy()

    num_features = df_temp.drop(columns=[col for col in df_temp.columns if 'alories' in col]).select_dtypes(include=['int64', 'float64']).columns

    if log == True:
        num_features = df_temp.drop(columns=['Calories', 'log_calories']).select_dtypes(include=['int64', 'float64']).columns
    
    cat_features = ['Sex_female']
    df_temp.drop(columns='Sex_male')

    num_only_int = []
    cat_num_int = []
    # numeric and numeric interactions
    for i in range(len(num_features)):
        for j in range(i+1, len(num_features)):
            df_temp[f"{num_features[i]}_{num_features[j]}"] = df_temp[num_features[i]] * df_temp[num_features[j]]
            num_only_int.append(f"{num_features[i]}_{num_features[j]}")

    # sex female and all numeric features
    for sex in cat_features:
        for i in range(len(num_features)):
            df_temp[f"{sex}_{num_features[i]}"] = df_temp[sex] * df_temp[num_features[i]]
            cat_num_int.append(f"{sex}_{num_features[i]}")

    return df_temp, num_only_int, cat_num_int

def make_features(df, test=False, make_log=True, make_int=True, use_pca=None):
    df_temp = df.copy()

    df_temp.drop(columns=['id'], inplace=True)

    # log transformations
    if make_log == True:
        df_temp = make_log_features(df_temp)
    
    # dummy encoding
    df_temp = pd.get_dummies(df_temp, columns=['Sex'])

    # new features, to be culled off in feature selection
    df_temp['BMI'] = df_temp['Weight'] / (df_temp['Height']/100)**2
    
    # make all interactions and stuff
    if make_int == True:
        df_temp, _, _ = make_interactions(df_temp)
    
    # apply KPCA for dimensionality reduction
    # if use_pca == 'kpca':
    #     kpca = KernelPCA(
    #         n_components = None,
    #         kernel = 'rbf',
    #         gamma = 10,
    #         alpha = 0.1
    #     )
    # elif use_pca == 'pca':
    #     pca = PCA()
        
        
    
    return df_temp

# for predicting log of outcome rather than just outcome
def make_features_log(df):
    df_temp = df.copy()

    df_temp = make_features(df_temp)

    df_temp['log_calories'] = np.log1p(df_temp['Calories'])
    
    return df_temp

df_train1 = make_features(df_train)
df_train2 = make_features_log(df_train)

# Model

## XGB Baseline

In [8]:
SEED = 30

In [9]:
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split

In [10]:
# without the fancy features
X = df_train.drop(columns=['id', 'Calories'])
X = pd.get_dummies(X, columns=['Sex'])
y = df_train['Calories']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=SEED)

# with the fancy features
X1 = df_train1.drop(columns=['Calories'])
y1 = df_train1['Calories']
X_train1, X_val1, _, _ = train_test_split(X1, y1, test_size=0.3, random_state=SEED)

# with log calories
X2 = df_train2.drop(columns=['Calories', 'log_calories'])
y2 = df_train2['log_calories']
X_train2, X_val2, y_train2, y_val2 = train_test_split(X2, y2, test_size=0.3, random_state=SEED)

In [11]:
# baseline without new features
xgb_baseline = xgb.XGBRegressor(enable_categorical=True)
xgb_baseline.fit(X_train, y_train)

y_val_pred = xgb_baseline.predict(X_val)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'XGB Baseline Score: {score}')

XGB Baseline Score: 0.06563999017507738


In [12]:
# baseline with new features
xgb_baseline1 = xgb.XGBRegressor(enable_categorical=True)
xgb_baseline1.fit(X_train1, y_train)

y_val_pred = xgb_baseline1.predict(X_val1)
y_val_pred = np.maximum(y_val_pred, 0)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'XGB Baseline Score With new Features: {score}')

XGB Baseline Score With new Features: 0.06572572306201457


In [13]:
# baseline with log calories
xgb_baseline2 = xgb.XGBRegressor(enable_categorical=True)
xgb_baseline2.fit(X_train2, y_train2)

y_val_pred = xgb_baseline2.predict(X_val2)
y_val_pred = np.expm1(y_val_pred)
y_val_pred = np.maximum(y_val_pred, 0)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'XGB Baseline Score With log Calories: {score}')

XGB Baseline Score With log Calories: 0.06288008971567988


In [14]:
# new features and PCA
scaler = StandardScaler()
X_train1_vector = scaler.fit_transform(X_train1)
X_val1_vector = scaler.transform(X_val1)

pca = PCA()
X_train3_vectors = pca.fit_transform(X_train1_vector)
cumulative_explained_variance = np.cumsum(pca.explained_variance_ratio_)
n_components = np.argmax(cumulative_explained_variance >= 0.95) + 1
print(n_components)
pca = PCA(n_components=n_components)

X_train3_vectors = pca.fit_transform(X_train1_vector)
X_val3_vectors = pca.transform(X_val1_vector)
X_train3 = pd.DataFrame(X_train3_vectors, columns=[f'PC_{i}' for i in range(X_train3_vectors.shape[1])])
X_val3 = pd.DataFrame(X_val3_vectors, columns=[f'PC_{i}' for i in range(X_train3_vectors.shape[1])])

xgb_baseline3 = xgb.XGBRegressor(enable_categorical=True)
xgb_baseline3.fit(X_train3, y_train)
y_val_pred = xgb_baseline3.predict(X_val3)
y_val_pred = np.maximum(y_val_pred, 0)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'PCA Score: {score}')

5
PCA Score: 0.0795513469096726


In [15]:
# PCA and log
scaler = StandardScaler()
X_train2_vector = scaler.fit_transform(X_train2)
X_val2_vector = scaler.transform(X_val2)

pca = PCA()
X_train3_vectors = pca.fit_transform(X_train2_vector)
cumulative_explained_variance = np.cumsum(pca.explained_variance_ratio_)
n_components = np.argmax(cumulative_explained_variance >= 0.95) + 1
print(n_components)
pca = PCA(n_components=n_components)

X_train3_vectors = pca.fit_transform(X_train2_vector)
X_val3_vectors = pca.transform(X_val2_vector)
X_train3 = pd.DataFrame(X_train3_vectors, columns=[f'PC_{i}' for i in range(X_train3_vectors.shape[1])])
X_val3 = pd.DataFrame(X_val3_vectors, columns=[f'PC_{i}' for i in range(X_train3_vectors.shape[1])])

xgb_baseline3 = xgb.XGBRegressor(enable_categorical=True)
xgb_baseline3.fit(X_train3, y_train2)
y_val_pred = xgb_baseline3.predict(X_val3)
y_val_pred = np.expm1(y_val_pred)
score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
print(f'PCA Score with log: {score}')

5
PCA Score with log: 0.07590267113658739


In [16]:
# # KPCA
# kpca = KernelPCA(
#     kernel = 'rbf',
#     n_components = 5
# )
# kpca.fit(X_train1_vector[:20000]) # not enough memory for entire X_train
# X_train4_vectors = kpca.transform(X_train1_vector)
# X_val4_vectors = kpca.transform(X_val1_vector)
# X_train4 = pd.DataFrame(X_train4_vectors, columns=[f'KPC_{i}' for i in range(X_train4_vectors.shape[1])])
# X_val4 = pd.DataFrame(X_val4_vectors, columns=[f'KPC_{i}' for i in range(X_val4_vectors.shape[1])])

# xgb_baseline4 = xgb.XGBRegressor(enable_categorical=True)
# xgb_baseline4.fit(X_train4, y_train)
# y_val_pred = xgb_baseline4.predict(X_val4)
# y_val_pred = np.maximum(0, y_val_pred)
# score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
# print(f"KPCA Score: {score}")

In [17]:
# KPCA with log
# kpca = KernelPCA(
#     kernel = 'rbf'
# )
# kpca.fit(X_train1_vector[:50000]) # not enough memory for entire X_train
# X_train4_vectors = kpca.transform(X_train1_vector)
# X_val4_vectors = kpca.transform(X_val1_vector)
# X_train4 = pd.DataFrame(X_train4_vectors, columns=[f'KPC_{i}' for i in range(X_train4_vectors.shape[1])])
# X_val4 = pd.DataFrame(X_val4_vectors, columns=[f'KPC_{i}' for i in range(X_val4_vectors.shape[1])])

# xgb_baseline5 = xgb.XGBRegressor(enable_categorical=True)
# xgb_baseline5.fit(X_train4, y_train2)
# y_val_pred = xgb_baseline4.predict(X_val4)
# y_val_pred = np.expm1(y_val_pred)
# score = np.sqrt(mean_squared_log_error(y_val_pred, y_val))
# print(f"KPCA Score with log calories: {score}")

## Big Tuna

In [18]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [19]:
def rmsle_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    y_pred = np.maximum(y_pred, 0)
    loss = np.sqrt(mean_squared_log_error(y_true, y_pred))
    return 'RMSLE', loss

# Function to run k-fold cross-validation with XGBoost and MSLE
def xgb_cv_rmsle(X, y, params, num_folds=5, debug=False, log=False, y_act=y):
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        y_val_act = y_act.iloc[val_idx]
        
        model = xgb.XGBRegressor(
            **params
        )
        model.fit(X_train,y_train)
        
        y_val_pred = model.predict(X_val)
        y_val_pred = np.maximum(0, y_val_pred)

        if log == True:
            y_val_pred = np.expm1(y_val_pred)
            
        score = np.sqrt(mean_squared_log_error(y_val_act, y_val_pred))

        if debug == True:
            print(score)
            
        fold_scores.append(score)
        
    return fold_scores

In [20]:
def objective(trial):
    params = {
        # "objective": "reg:squarederror",
        "eval_metric" : "rmse",
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, step=0.01),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1),
        "max_bin": trial.suggest_int("max_bin", 256, 2048),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 0.1, step=0.01),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "n_estimators": trial.suggest_int("n_estimators", 50, 2000),
        "max_delta_step": trial.suggest_int("max_delta_step", 1, 10),
        "random_state": SEED
    }

    score = np.mean(xgb_cv_rmsle(X=X2, y=y2, params=params, debug=False, log=True))
    return score

In [21]:
# %%time
# study = optuna.create_study(direction='minimize',
#                             sampler = optuna.samplers.RandomSampler(seed=SEED),
#                             study_name = "BIG BLUE FIN TUNA!!")
# study.optimize(objective, n_trials=100, show_progress_bar=True, )

In [22]:
# best_params = study.best_params
# print(f'Best Trial Params: {best_params}')

# print(f'Best Trial Value: {study.best_trial.value}')

In [23]:
# # for saving versions
best_params = {'learning_rate': 0.03, 'max_depth': 14, 'subsample': 1.0, 'colsample_bytree': 0.7, 'max_bin': 1774, 'min_child_weight': 4, 'gamma': 0.02, 'lambda': 0.5720902445525685, 'alpha': 4.286145035324733, 'grow_policy': 'lossguide', 'n_estimators': 546, 'max_delta_step': 2}

# Submission

In [24]:
best_model = xgb.XGBRegressor(**best_params)
best_model.fit(X_train3, y_train2)

XGBRegressor(alpha=4.286145035324733, base_score=None, booster=None,
             callbacks=None, colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=0.02, grow_policy='lossguide', importance_type=None,
             interaction_constraints=None, lambda=0.5720902445525685,
             learning_rate=0.03, max_bin=1774, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=2, max_depth=14,
             max_leaves=None, min_child_weight=4, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=546,
             n_jobs=None, ...)

In [25]:
y_val_pred = best_model.predict(X_val3)
y_val_pred = np.expm1(y_val_pred)
score = mean_squared_log_error(y_val_pred, y_val)
print(score)

0.006002606342366172


In [26]:
# X_train2.head()

In [27]:
df_test1 = make_features(df_test, test=True)
df_test1 = scaler.transform(df_test1)

df_test_vectors = pca.transform(df_test1)
df_test1 = pd.DataFrame(df_test_vectors, columns=[f'PC_{i}' for i in range(df_test_vectors.shape[1])])

df_test1.head()

,PC_0,PC_1,PC_2,PC_3,PC_4
0,0.207983,6.367281,1.274064,2.141100,0.190653
1,8.385060,2.136872,-6.201417,-2.951218,-1.425390
2,1.984056,-1.631518,-3.401173,-5.649169,2.108171
3,1.496666,-5.394529,-0.180882,-2.054688,2.622169
4,-3.960914,-3.318974,-2.629091,-2.579095,-0.827454


In [28]:
y_test_pred = best_model.predict(df_test1)
y_test_pred = np.expm1(y_test_pred)

submission = pd.read_csv("/kaggle/input/playground-series-s5e5/sample_submission.csv")
submission['Calories'] = y_test_pred
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Calories
0,750000,28.243109
1,750001,106.825981
2,750002,89.062126
3,750003,126.167519
4,750004,74.844612


# TESTING SHIT

## ADD THIS SOMEWHERE SOMEHOW THIS THING MAYBE IS KINDA EPIC

#